# Chapter 4.5: Cost Tradeoffs and Choosing the Right Tool

Goal: Calculate LLM costs for real scenarios, apply a decision framework for choosing between extraction approaches, and understand production considerations like caching and ROI.

### Topics:
- Token estimation and cost calculation
- Comparing costs across LLM providers and models
- Decision framework: LLM vs regex vs traditional ML
- Caching strategies to reduce costs
- ROI analysis for LLM-based pipelines
- When NOT to use an LLM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Quick Recap

- **Token**: The basic unit LLMs process — roughly 4 characters or 0.75 words in English
- **Input tokens**: Tokens in your prompt (you pay for these)
- **Output tokens**: Tokens the LLM generates (usually more expensive per token)
- **Cost per token**: Varies by model — cheaper models are faster but less capable
- **Caching**: Storing results to avoid re-processing identical inputs
- **ROI (Return on Investment)**: Whether the value gained exceeds the cost

## Data

All data for this activity is defined inline — pricing tables, task scenarios, and cost parameters.

In [ ]:
# LLM pricing table (per 1 million tokens, approximate as of early 2026)
pricing = {
    "GPT-4o": {"input": 2.50, "output": 10.00},
    "GPT-4o-mini": {"input": 0.15, "output": 0.60},
    "Claude Sonnet": {"input": 3.00, "output": 15.00},
    "Gemini Flash": {"input": 0.10, "output": 0.40},
}

# Display as a nice table
pricing_df = pd.DataFrame(pricing).T
pricing_df.columns = ["Input ($/1M tokens)", "Output ($/1M tokens)"]
pricing_df

In [ ]:
# Task scenarios for the decision framework exercise
scenarios = [
    {
        "id": 1,
        "task": "Extract email addresses from 50,000 customer support tickets",
        "complexity": "low",
        "volume": 50000,
        "accuracy_needed": "99%+"
    },
    {
        "id": 2,
        "task": "Classify customer complaints into 25 categories with subcategories",
        "complexity": "high",
        "volume": 10000,
        "accuracy_needed": "85%+"
    },
    {
        "id": 3,
        "task": "Detect whether a product review mentions a safety issue",
        "complexity": "medium",
        "volume": 100000,
        "accuracy_needed": "95%+ (safety critical)"
    },
    {
        "id": 4,
        "task": "Extract phone numbers from business listings",
        "complexity": "low",
        "volume": 200000,
        "accuracy_needed": "99%+"
    },
    {
        "id": 5,
        "task": "Summarize 500 legal contracts into key terms and obligations",
        "complexity": "very high",
        "volume": 500,
        "accuracy_needed": "90%+ (legal review)"
    },
    {
        "id": 6,
        "task": "Determine sentiment of 1 million tweets about your brand",
        "complexity": "medium",
        "volume": 1000000,
        "accuracy_needed": "80%+"
    }
]

## Practice

### 1. By hand — Write a token estimation function

A common rule of thumb: **1 token ≈ 4 characters** (for English text). Write a function `estimate_tokens(text)` that estimates the number of tokens in a string. Test it on strings of various lengths.

In [ ]:
def estimate_tokens(text):
    """Estimate the number of tokens in a text string.
    
    Rule of thumb: 1 token ≈ 4 characters for English text.
    """
    ...

# Test on various strings
test_strings = [
    "Hello world",
    "This is a simple product review.",
    "Absolutely love this coffee maker! Best purchase I've made all year. Brews perfect coffee every morning.",
    "A" * 1000,  # 1000-character string
]

for s in test_strings:
    tokens = estimate_tokens(s)
    print(f"'{s[:50]}{'...' if len(s) > 50 else ''}' → {len(s)} chars → ~{tokens} tokens")

### 2. By hand — Calculate sentiment extraction costs

You need to extract sentiment from **10,000 product reviews**. Each review averages **150 characters** (~38 tokens). Your prompt template adds **200 tokens** of instructions. The expected output is **50 tokens** (JSON response).

Calculate the total cost for each of the 4 models in our pricing table.

In [ ]:
# Parameters
num_reviews = 10000
tokens_per_review = 38       # ~150 characters
prompt_overhead = 200        # instruction tokens
output_tokens_per_call = 50  # JSON response

# Total tokens per API call
input_tokens_per_call = tokens_per_review + prompt_overhead

# Calculate costs for each model
sentiment_costs = {}
for model_name, prices in pricing.items():
    # Cost = (input_tokens * input_price + output_tokens * output_price) * num_calls
    # Remember: prices are per 1 MILLION tokens
    input_cost = ...
    output_cost = ...
    total_cost = ...
    sentiment_costs[model_name] = total_cost
    print(f"{model_name}: ${total_cost:.2f}")

### 3. By hand — Calculate complex extraction costs

Now calculate costs for a more complex task: extracting **5 fields** (title, category, experience_level, remote_policy, skills) from **5,000 job postings**. Each posting averages **400 characters** (~100 tokens). Your prompt is longer: **400 tokens** (includes field descriptions and valid values). Expected output: **120 tokens** (larger JSON with a list field).

In [ ]:
# Parameters for complex extraction
num_postings = 5000
tokens_per_posting = 100
prompt_overhead_complex = 400
output_tokens_complex = 120

input_tokens_complex = tokens_per_posting + prompt_overhead_complex

# Calculate costs for each model
complex_costs = {}
for model_name, prices in pricing.items():
    input_cost = ...
    output_cost = ...
    total_cost = ...
    complex_costs[model_name] = total_cost
    print(f"{model_name}: ${total_cost:.2f}")

### 4. Use AI — Visualize cost comparison

Use your AI assistant to create a grouped bar chart that shows both the sentiment extraction costs and the complex extraction costs side by side for each model. Use different colors for each task type.

In [ ]:
# Use AI to create the grouped bar chart
# Combine sentiment_costs and complex_costs into a visualization
...

**Your observation:** How many times more expensive is GPT-4o compared to Gemini Flash? When might the extra cost be justified?

(Write your answer here)

### 5. By hand — Decision framework

For each of the 6 scenarios below, choose the best approach: **regex**, **traditional ML** (trained classifier), or **LLM**. Justify your choice based on complexity, volume, accuracy needs, and cost.

In [ ]:
# Display the scenarios
for s in scenarios:
    print(f"Scenario {s['id']}: {s['task']}")
    print(f"  Complexity: {s['complexity']} | Volume: {s['volume']:,} | Accuracy needed: {s['accuracy_needed']}")
    print()

**Your decisions:**

Scenario 1 — Extract email addresses (50K tickets): 
Choice: _______ | Reason: 

(Write your answer here)

Scenario 2 — Classify complaints into 25 categories (10K): 
Choice: _______ | Reason: 

(Write your answer here)

Scenario 3 — Detect safety issues (100K reviews): 
Choice: _______ | Reason: 

(Write your answer here)

Scenario 4 — Extract phone numbers (200K listings): 
Choice: _______ | Reason: 

(Write your answer here)

Scenario 5 — Summarize legal contracts (500): 
Choice: _______ | Reason: 

(Write your answer here)

Scenario 6 — Tweet sentiment (1M tweets): 
Choice: _______ | Reason: 

(Write your answer here)

### 6. Use AI — Implement a caching function

Use your AI assistant to write a `cached_extract()` function that:
1. Takes a text and a cache dictionary
2. If the text is already in the cache, returns the cached result (and prints "cache hit")
3. If not, simulates an API call (just returns a placeholder dict) and stores it in the cache

Test it with some duplicate texts to show the cache working.

In [ ]:
# Use AI to implement the caching function
def cached_extract(text, cache):
    """Extract features from text, using cache for duplicates.
    
    Args:
        text: The input text to process
        cache: Dictionary mapping text -> extraction result
    
    Returns:
        Extraction result (from cache or freshly computed)
    """
    ...

# Test with duplicate texts
test_texts = [
    "Great product, love it!",
    "Terrible quality, broke immediately.",
    "Great product, love it!",       # duplicate
    "It's okay, nothing special.",
    "Terrible quality, broke immediately.",  # duplicate
    "Great product, love it!",       # duplicate again
]

cache = {}
for text in test_texts:
    result = cached_extract(text, cache)

print(f"\nTotal calls: {len(test_texts)}")
print(f"Cache size: {len(cache)}")
print(f"API calls saved: {len(test_texts) - len(cache)}")

### 7. By hand — ROI calculation

Your company currently has humans classify customer support tickets. You're evaluating whether to switch to an LLM.

**Current process (human):**
- Cost: $0.50 per ticket
- Accuracy: 95%
- Speed: 2 minutes per ticket

**Proposed process (LLM):**
- Cost: $0.001 per ticket
- Accuracy: 90%
- Speed: 0.5 seconds per ticket

**Misclassification cost:** Each incorrectly classified ticket costs the company $5.00 on average (wrong department, delayed resolution, unhappy customer).

Calculate the total cost per 10,000 tickets for each approach.

In [ ]:
num_tickets = 10000

# Human approach
human_processing_cost = ...  # $0.50 * 10,000
human_error_rate = ...       # 1 - 0.95
human_errors = ...           # error_rate * num_tickets
human_error_cost = ...       # errors * $5.00
human_total = ...            # processing + error cost

print(f"=== Human Approach ===")
print(f"Processing cost: ${human_processing_cost:,.2f}")
print(f"Misclassification cost: ${human_error_cost:,.2f} ({human_errors:.0f} errors)")
print(f"Total: ${human_total:,.2f}")
print()

# LLM approach
llm_processing_cost = ...    # $0.001 * 10,000
llm_error_rate = ...         # 1 - 0.90
llm_errors = ...             # error_rate * num_tickets
llm_error_cost = ...         # errors * $5.00
llm_total = ...              # processing + error cost

print(f"=== LLM Approach ===")
print(f"Processing cost: ${llm_processing_cost:,.2f}")
print(f"Misclassification cost: ${llm_error_cost:,.2f} ({llm_errors:.0f} errors)")
print(f"Total: ${llm_total:,.2f}")
print()

print(f"=== Comparison ===")
savings = human_total - llm_total
print(f"Savings with LLM: ${savings:,.2f}")

**Your analysis:** Which approach is cheaper overall? At what misclassification cost would the LLM approach become MORE expensive than the human approach? (Hint: set the totals equal and solve for the misclassification cost.)

(Write your answer here)

## Discussion

Why is "just use GPT-4 for everything" wrong? Give at least 3 concrete reasons with examples.

(Discuss with a neighbor)